In [12]:
# !pip install -U llama-index crewai langchain-groq llama-index-embeddings-huggingface
# !pip install --upgrade transformers sentence-transformers
# !pip install llama-index-embeddings-fastembed
# !pip install fastembed fastembed-gpu 
!pip install llama-index-llms-langchain

In [14]:
import os
from langchain_groq import ChatGroq
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from crewai.tasks.task_output import TaskOutput


groq_llm = ChatGroq(
    model="openai/gpt-oss-20b", 
    temperature=0, 
    api_key="YOUR_GROQ_API_KEY"
)

Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = groq_llm 
documents = SimpleDirectoryReader(input_files=["/kaggle/input/datasets/sathishkumar132/samplepdf/pdf.pdf"]).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

2026-02-21 17:25:16,673 - INFO - NumExpr defaulting to 4 threads.


In [ ]:
@tool("PDF Search Tool")
def pdf_search_tool(query: str) -> str:
    """sample"""
    response = query_engine.query(query)
    return str(response)

def print_task_status(task_output: TaskOutput):
    print(f"[Callback Triggered] Task Finished! Output preview: {task_output.raw[:100]}")

researcher = Agent(
    role='PDF Document Researcher',
    goal='Extract precise, factual information from the PDF using the search tool.',
    backstory='An analytical expert who finds needles in document haystacks.',
    tools=[pdf_search_tool],
    llm=groq_llm,
    verbose=True
)

writer = Agent(
    role='Senior Synthesizer',
    goal='Take raw research and turn it into a concise, readable summary.',
    backstory='A skilled technical writer who transforms complex data into plain English.',
    llm=groq_llm,
    verbose=True
)

task_research = Task(
    description='Search the PDF document for the main conclusions and core themes.',
    expected_output='A bulleted list of the top 3 core conclusions found in the PDF.',
    agent=researcher,
    callback=print_task_status
)

task_write = Task(
    description='Using the researcher\'s bullet points, write a 2-paragraph final report.',
    expected_output='A clean, 2-paragraph summary report.',
    agent=writer,
    callback=print_task_status
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    process=Process.sequential, 
    memory=True, 
    verbose=True
)

result = crew.kickoff()

print(result)